In [1]:
import requests
import pandas as pd
from tqdm import tqdm
import time
import os

# 1. Data Retrieval

In [2]:
# STEP 1 — API Setup

API_KEY = 'oe_QGgWdwVTERbwcSmN5Nhwnt'

BASE_URL = "https://api.openelectricity.org.au/v4"

NETWORK = "NEM"

STATUS = "operating"

INTERVAL = "5m"

START_DATE = "2026-05-09T00:00:00"

END_DATE = "2026-05-16T00:00:00"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}"
}

In [3]:
# STEP 2 — Get Facilities List

facility_url = f"{BASE_URL}/facilities/"

facility_params = {
    "network_id": NETWORK,
    "status_id": STATUS
}

facility_response = requests.get(
    facility_url,
    headers=HEADERS,
    params=facility_params
)

print("Status Code:", facility_response.status_code)


# Convert JSON

facility_json = facility_response.json()


# Extract data

facility_data = facility_json["data"]


# Check total facilities

print("Total Facilities:", len(facility_data))


# View first facility

facility_data[0]

Status Code: 200
Total Facilities: 432


{'code': 'ADP',
 'code_display': 'ADP',
 'name': 'Adelaide Desalination',
 'network_id': 'NEM',
 'network_region': 'SA1',
 'description': '<p>The Adelaide Desalination plant (ADP), formerly known as the Port Stanvac Desalination Plant, is a sea water reverse osmosis desalination plant located in Lonsdale, South Australia which has the capacity to provide the city of Adelaide with up to 50% of its drinking water needs.</p>',
 'osm_way_id': '12798948',
 'location': {'lat': -35.100751, 'lng': 138.483357},
 'units': [{'code': 'ADPBA1G',
   'code_display': '0ADPBA1G',
   'fueltech_id': 'battery_discharging',
   'status_id': 'operating',
   'capacity_registered': 7.76,
   'capacity_maximum': 6.15,
   'capacity_storage': 12.6,
   'data_first_seen': '2021-05-18T10:55:00+10:00',
   'data_last_seen': '2026-05-17T03:20:00+10:00',
   'dispatch_type': 'GENERATOR',
   'commencement_date': '2021-05-17T14:00:00+10:00',
   'commencement_date_specificity': 'day',
   'closure_date_specificity': 'day',
  

In [4]:
# STEP 3 — Create Empty Data Dictionary

data_dict = {

    "facility_name": [],

    "facility_code": [],

    "unit_code": [],

    "power_mw": [],

    "emissions_t": [],

    "timestamp": [],

    "lat": [],

    "lon": []
}

In [5]:
# STEP 4 Exploration / Structure Inspection
first_facility = facility_data[0]

first_facility

{'code': 'ADP',
 'code_display': 'ADP',
 'name': 'Adelaide Desalination',
 'network_id': 'NEM',
 'network_region': 'SA1',
 'description': '<p>The Adelaide Desalination plant (ADP), formerly known as the Port Stanvac Desalination Plant, is a sea water reverse osmosis desalination plant located in Lonsdale, South Australia which has the capacity to provide the city of Adelaide with up to 50% of its drinking water needs.</p>',
 'osm_way_id': '12798948',
 'location': {'lat': -35.100751, 'lng': 138.483357},
 'units': [{'code': 'ADPBA1G',
   'code_display': '0ADPBA1G',
   'fueltech_id': 'battery_discharging',
   'status_id': 'operating',
   'capacity_registered': 7.76,
   'capacity_maximum': 6.15,
   'capacity_storage': 12.6,
   'data_first_seen': '2021-05-18T10:55:00+10:00',
   'data_last_seen': '2026-05-17T03:20:00+10:00',
   'dispatch_type': 'GENERATOR',
   'commencement_date': '2021-05-17T14:00:00+10:00',
   'commencement_date_specificity': 'day',
   'closure_date_specificity': 'day',
  

In [6]:
# STEP 6 — Request  Request Power + Emissions Data

for facility in facility_data:

    # Facility Information
    facility_code = facility["code"]

    facility_name = facility["name"]

    lat = facility["location"]["lat"]

    lon = facility["location"]["lng"]

    # API URL
    data_url = f"{BASE_URL}/data/facilities/{NETWORK}"
    
    # Request Power Data
    power_params = {
        "metrics": ["power"],
        "interval": INTERVAL,
        "facility_code": facility_code,
        "date_start": START_DATE,
        "date_end": END_DATE
    }

    power_response = requests.get(
        data_url,
        headers=HEADERS,
        params=power_params
    )

    power_json = power_response.json()

    
    # Request Emissions Data
    emissions_params = {
        "metrics": ["emissions"],
        "interval": INTERVAL,
        "facility_code": facility_code,
        "date_start": START_DATE,
        "date_end": END_DATE
    }

    emissions_response = requests.get(
        data_url,
        headers=HEADERS,
        params=emissions_params
    )

    emissions_json = emissions_response.json()

    
    # STEP 7 — Extract Power + Emissions
    
    # Error Handling
    error_codes = [400, 401, 403, 404, 416, 422, 500]

    if power_response.status_code in error_codes:
        continue

    if emissions_response.status_code in error_codes:
        continue

    power_data = power_json["data"][0]["results"][0]

    emissions_data = emissions_json["data"][0]["results"][0]

    
    # STEP 8 — Integrate Facility Data
    
    for power_row, emissions_row in zip(
        power_data["data"],
        emissions_data["data"]
    ):

        timestamp = power_row[0]

        power_value = power_row[1]

        emissions_value = emissions_row[1]

        data_dict["facility_name"].append(
            facility_name
        )

        data_dict["facility_code"].append(
            facility_code
        )

        data_dict["unit_code"].append(
            power_data["columns"]["unit_code"]
        )

        data_dict["power_mw"].append(
            power_value
        )

        data_dict["emissions_t"].append(
            emissions_value
        )

        data_dict["timestamp"].append(
            timestamp
        )

        data_dict["lat"].append(
            lat
        )

        data_dict["lon"].append(
            lon
        )

In [7]:
# STEP 7 — Create DataFrame
df = pd.DataFrame(data_dict)

df.head()
df["facility_code"].nunique()

353

In [8]:
# STEP 11 — Save CSV

file_name = "raw_data.csv"

if os.path.exists(file_name):

    print("CSV file already exists.")

else:

    df.to_csv(
        file_name,
        index=False
    )

In [9]:
df["emissions_t"].describe()

count    710385.000000
mean          1.201308
std           6.428755
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          54.834900
Name: emissions_t, dtype: float64

# 2. Data Integration and Materialisation/Caching

In [10]:
# STEP 12 — Load Raw Data
df = pd.read_csv("raw_data.csv")

df.head()

,facility_name,facility_code,unit_code,power_mw,emissions_t,timestamp,lat,lon
0,Adelaide Desalination,ADP,ADPBA1,-0.005,0.0,2026-05-09T00:00:00+10:00,-35.100751,138.483357
1,Adelaide Desalination,ADP,ADPBA1,0.027,0.0,2026-05-09T00:05:00+10:00,-35.100751,138.483357
2,Adelaide Desalination,ADP,ADPBA1,0.008,0.0,2026-05-09T00:10:00+10:00,-35.100751,138.483357
3,Adelaide Desalination,ADP,ADPBA1,0.071,0.0,2026-05-09T00:15:00+10:00,-35.100751,138.483357
4,Adelaide Desalination,ADP,ADPBA1,0.009,0.0,2026-05-09T00:20:00+10:00,-35.100751,138.483357


In [11]:
# STEP 13 — Basic Preprocessing
# Remove missing values
df = df.dropna()

# Remove duplicate rows
df = df.drop_duplicates()

# Convert timestamp datatype
df["timestamp"] = pd.to_datetime(
    df["timestamp"]
)

df.head()

,facility_name,facility_code,unit_code,power_mw,emissions_t,timestamp,lat,lon
0,Adelaide Desalination,ADP,ADPBA1,-0.005,0.0,2026-05-09 00:00:00+10:00,-35.100751,138.483357
1,Adelaide Desalination,ADP,ADPBA1,0.027,0.0,2026-05-09 00:05:00+10:00,-35.100751,138.483357
2,Adelaide Desalination,ADP,ADPBA1,0.008,0.0,2026-05-09 00:10:00+10:00,-35.100751,138.483357
3,Adelaide Desalination,ADP,ADPBA1,0.071,0.0,2026-05-09 00:15:00+10:00,-35.100751,138.483357
4,Adelaide Desalination,ADP,ADPBA1,0.009,0.0,2026-05-09 00:20:00+10:00,-35.100751,138.483357


In [12]:
# STEP 14 — Sort Data by Time
df = df.sort_values(
    by="timestamp"
)

df = df.reset_index(drop=True)

df.head()

,facility_name,facility_code,unit_code,power_mw,emissions_t,timestamp,lat,lon
0,Adelaide Desalination,ADP,ADPBA1,-0.0050,0.0,2026-05-09 00:00:00+10:00,-35.100751,138.483357
1,Metz N,METZSF,METZSF1,0.0000,0.0,2026-05-09 00:00:00+10:00,-30.515484,151.859815
2,Melbourne A3,MREHA3,MREHA3,-4.0452,0.0,2026-05-09 00:00:00+10:00,-37.663819,144.726859
3,Melbourne A2,MREHA2,MREHA2,-1.5811,0.0,2026-05-09 00:00:00+10:00,-37.663934,144.726927
4,Melbourne A1,MREH,MREHA1,-1.4469,0.0,2026-05-09 00:00:00+10:00,-37.661157,144.726292


In [13]:
# STEP 15 — Materialisation / Caching

file_name = "integrated_power_emissions_data.csv"

if os.path.exists(file_name):

    print("CSV file already exists.")

else:

    df.to_csv(
        file_name,
        index=False
    )

    print("Integrated CSV saved successfully.")

Integrated CSV saved successfully.
